In [1]:
%pip install xgboost sentence-transformers shap lime torch gensim scikit-learn pandas numpy matplotlib seaborn tqdm nltk tf-keras

  Installing build dependencies ...   Installing build dependencies ... -done
  Getting requirements to build wheel ... one
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/115.9 MB ? eta -:--:--Downloading xgboost-3.1.2-py3-none-manylinux_2_28_x86_64.whl (115.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 MB 125.2 MB/s  0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 MB 125.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 126.7 MB/s  0:00:0090m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.0 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 126.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/566.1 kB ? eta -:--:--Downloading huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 13.4 

# AI Design Pattern Classification Research Notebook

This notebook implements a comprehensive pipeline for classifying AI Design Patterns using various embedding techniques and machine learning/deep learning models.

## 1. Setup and Imports
We will install necessary libraries and import them.

In [2]:
import os
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import xgboost as xgb

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Embeddings
from gensim.models import Word2Vec, KeyedVectors
import gensim.downloader as api
from sentence_transformers import SentenceTransformer

# Explainability
import shap
import lime
from lime import lime_text

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)
torch.manual_seed(42)

print("Libraries imported successfully.")

/root/AI-Pattern-Mining-Project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-02 08:04:44.287229: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-02 08:04:44.287229: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Libraries imported successfully.


In [3]:
# Define Dataset Path
DATASET_PATH = "/root/AI-Pattern-Mining-Project/outputs/prompt & rag/20251028_085918 - Run/generated_code-v2"

def load_dataset(root_path):
    data = []
    labels = []
    
    if not os.path.exists(root_path):
        print(f"Error: Path {root_path} does not exist.")
        return pd.DataFrame()

    # Walk through the directory
    for label in os.listdir(root_path):
        label_path = os.path.join(root_path, label)
        if os.path.isdir(label_path):
            for file_name in os.listdir(label_path):
                file_path = os.path.join(label_path, file_name)
                if os.path.isfile(file_path):
                    try:
                        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                            content = f.read()
                            if content.strip(): # Skip empty files
                                data.append(content)
                                labels.append(label)
                    except Exception as e:
                        print(f"Error reading {file_path}: {e}")
    
    df = pd.DataFrame({'code': data, 'label': labels})
    return df

# Load the data
print("Loading dataset...")
df = load_dataset(DATASET_PATH)
print(f"Dataset loaded. Shape: {df.shape}")
print("Class distribution:")
print(df['label'].value_counts())
df.head()

Loading dataset...
Dataset loaded. Shape: (1430, 2)
Class distribution:
label
Explainable AI (XAI) Techniques                                                                 97
Enhanced User Intent Comprehension with LLMs                                                    97
Tool Use for LLMs                                                                               97
Structured Output & Formatting for LLMs                                                         97
LLM Agent Training & Alignment                                                                  97
LLM Results Evaluation                                                                          97
LLM KV Cache Optimization                                                                       97
Modular LLM Agent Architectures                                                                 97
LLMs for Recommender Systems                                                                    97
LLM based Planning, Iterative O

,code,label
0,from pydantic import BaseModel\nimport json\n\...,Advanced LLM Prompting
1,import random\n\nclass MedicalDiagnosticAssist...,Advanced LLM Prompting
2,import gradio as gr\nfrom transformers import ...,Advanced LLM Prompting
3,import streamlit as st\nimport os\nfrom dotenv...,Advanced LLM Prompting
4,class AutomatedPromptOptimizer:\n def __ini...,Advanced LLM Prompting


In [4]:
import re
import nltk
from nltk.tokenize import word_tokenize

# Download nltk resources if not present
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

def preprocess_code(code):
    # Remove comments (simple regex for C-style and Python-style)
    code = re.sub(r'//.*', '', code)
    code = re.sub(r'#.*', '', code)
    code = re.sub(r'/\*[\s\S]*?\*/', '', code)
    
    # Remove special characters but keep structure relevant ones if needed, 
    # but for general embedding, we often clean up.
    # Let's keep it simple: alphanumeric and some symbols
    code = re.sub(r'[^a-zA-Z0-9\s_]', ' ', code)
    
    # Collapse whitespace
    code = re.sub(r'\s+', ' ', code).strip()
    return code

df['cleaned_code'] = df['code'].apply(preprocess_code)
print("Preprocessing complete.")
df[['code', 'cleaned_code']].head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data]   Unzipping tokenizers/punkt.zip.


Preprocessing complete.


,code,cleaned_code
0,from pydantic import BaseModel\nimport json\n\...,from pydantic import BaseModel import json cla...
1,import random\n\nclass MedicalDiagnosticAssist...,import random class MedicalDiagnosticAssistant...
2,import gradio as gr\nfrom transformers import ...,import gradio as gr from transformers import p...
3,import streamlit as st\nimport os\nfrom dotenv...,import streamlit as st import os from dotenv i...
4,class AutomatedPromptOptimizer:\n def __ini...,class AutomatedPromptOptimizer def __init__ se...


In [5]:
# 1. TF-IDF
print("Generating TF-IDF Embeddings...")
tfidf_vectorizer = TfidfVectorizer(max_features=2000)
X_tfidf = tfidf_vectorizer.fit_transform(df['cleaned_code']).toarray()
print(f"TF-IDF Shape: {X_tfidf.shape}")

# 2. Word2Vec (Trained on this dataset)
print("Training Word2Vec...")
tokenized_code = [code.split() for code in df['cleaned_code']]
w2v_model = Word2Vec(sentences=tokenized_code, vector_size=100, window=5, min_count=1, workers=4)

def get_avg_w2v(tokens, model, vector_size):
    # Check if model is KeyedVectors or Word2Vec
    if hasattr(model, 'wv'):
        wv = model.wv
    else:
        wv = model
        
    valid_words = [word for word in tokens if word in wv]
    if not valid_words:
        return np.zeros(vector_size)
    return np.mean(wv[valid_words], axis=0)

X_w2v = np.array([get_avg_w2v(tokens, w2v_model, 100) for tokens in tokenized_code])
print(f"Word2Vec Shape: {X_w2v.shape}")

# 3. GloVe (Using Pre-trained if available, else skip or mock)
# We will use a small pre-trained model from gensim-data if possible
print("Loading GloVe (glove-wiki-gigaword-50)...")
try:
    glove_model = api.load("glove-wiki-gigaword-50")
    X_glove = np.array([get_avg_w2v(tokens, glove_model, 50) for tokens in tokenized_code])
    print(f"GloVe Shape: {X_glove.shape}")
except Exception as e:
    print(f"Could not load GloVe: {e}")
    X_glove = None

Generating TF-IDF Embeddings...
TF-IDF Shape: (1430, 2000)
Training Word2Vec...
TF-IDF Shape: (1430, 2000)
Training Word2Vec...
Word2Vec Shape: (1430, 100)
Loading GloVe (glove-wiki-gigaword-50)...
Word2Vec Shape: (1430, 100)
Loading GloVe (glove-wiki-gigaword-50)...
[==================================================] 100.0% 66.0/66.0MB downloaded

GloVe Shape: (1430, 50)
GloVe Shape: (1430, 50)


In [9]:
from transformers import AutoTokenizer, AutoModel
import torch

embeddings_dict = {
    'TF-IDF': X_tfidf,
    'Word2Vec': X_w2v
}
if X_glove is not None:
    embeddings_dict['GloVe'] = X_glove

# Helper function using Transformers directly
def get_hf_embeddings(model_name, texts, batch_size=16, trust_remote_code=False):
    print(f"Generating embeddings for {model_name}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=trust_remote_code)
        model = AutoModel.from_pretrained(model_name, trust_remote_code=trust_remote_code)
        
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        model.eval()
        
        all_embeddings = []
        
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            
            # Tokenize
            inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt", max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = model(**inputs)
                # Mean pooling
                if hasattr(outputs, 'last_hidden_state'):
                    embeddings = outputs.last_hidden_state.mean(dim=1)
                else:
                    # Fallback for some models
                    embeddings = outputs[0].mean(dim=1)
                    
            all_embeddings.append(embeddings.cpu().numpy())
            
        return np.vstack(all_embeddings)
        
    except Exception as e:
        print(f"Failed to load/run {model_name}: {e}")
        return None

# List of models to try
# We will run on the full dataset (1430 samples)

# 4. Sentence-BERT (Generic)
sbert_emb = get_hf_embeddings('sentence-transformers/all-MiniLM-L6-v2', df['cleaned_code'].tolist())
if sbert_emb is not None:
    embeddings_dict['Sentence-BERT'] = sbert_emb

# 5. CodeBERT (Microsoft)
codebert_emb = get_hf_embeddings('microsoft/codebert-base', df['cleaned_code'].tolist())
if codebert_emb is not None:
    embeddings_dict['CodeBERT'] = codebert_emb

# 6. RoBERTa
roberta_emb = get_hf_embeddings('roberta-base', df['cleaned_code'].tolist())
if roberta_emb is not None:
    embeddings_dict['RoBERTa'] = roberta_emb

# 7. Jina Embeddings (jinaai/jina-embeddings-v2-base-code)
jina_emb = get_hf_embeddings('jinaai/jina-embeddings-v2-base-code', df['cleaned_code'].tolist(), trust_remote_code=True)
if jina_emb is not None:
    embeddings_dict['Jina-V2'] = jina_emb

# 8. Nomic Embed Code (nomic-ai/nomic-embed-text-v1)
nomic_texts = ["search_document: " + t for t in df['cleaned_code'].tolist()]
nomic_emb = get_hf_embeddings('nomic-ai/nomic-embed-text-v1', nomic_texts, trust_remote_code=True)
if nomic_emb is not None:
    embeddings_dict['Nomic'] = nomic_emb

# Check shapes
for name, emb in embeddings_dict.items():
    print(f"{name}: {emb.shape}")

Generating embeddings for sentence-transformers/all-MiniLM-L6-v2...


Generating embeddings for microsoft/codebert-base...
Generating embeddings for roberta-base...
Generating embeddings for roberta-base...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Generating embeddings for jinaai/jina-embeddings-v2-base-code...


A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-bert-v2-qk-post-norm:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-bert-v2-qk-post-norm:
- modeling_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-bert-v2-qk-post-norm:
- modeling_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Generating embeddings for nomic-ai/nomic-embed-text-v1...


A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- configuration_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- modeling_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- modeling_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
<All keys matched successfully>
<All keys matched successfully>


TF-IDF: (1430, 2000)
Word2Vec: (1430, 100)
GloVe: (1430, 50)
Sentence-BERT: (1430, 384)
CodeBERT: (1430, 768)
RoBERTa: (1430, 768)
Jina-V2: (1430, 768)
Nomic: (1430, 768)


In [10]:
def evaluate_models(X, y, embedding_name):
    results = []
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Naive Bayes': GaussianNB(), # GaussianNB handles negative values better than MultinomialNB for some embeddings
        'SVM': SVC(probability=True),
        'Random Forest': RandomForestClassifier(),
        'Gradient Boosting': GradientBoostingClassifier(),
        'XGBoost': xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'),
        'KNN': KNeighborsClassifier()
    }
    
    # Encode labels for XGBoost
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_test_enc = le.transform(y_test)
    
    for name, model in models.items():
        start_time = time.time()
        
        # Handle XGBoost label encoding
        if name == 'XGBoost':
            model.fit(X_train, y_train_enc)
            y_pred_enc = model.predict(X_test)
            y_pred = le.inverse_transform(y_pred_enc)
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            
        end_time = time.time()
        duration = end_time - start_time
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        
        results.append({
            'Model': name,
            'Embedding': embedding_name,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-score': f1,
            'Time (s)': duration
        })
        
    return pd.DataFrame(results)

# Run evaluation for all embeddings
all_results = []
labels = df['label']

print("Starting Model Evaluation...")
for emb_name, emb_data in embeddings_dict.items():
    print(f"Evaluating models with {emb_name}...")
    # Ensure no NaNs
    if np.isnan(emb_data).any():
        emb_data = np.nan_to_num(emb_data)
        
    # Naive Bayes requires non-negative for Multinomial, but we used Gaussian. 
    # However, some embeddings might be negative. Gaussian is fine.
    
    res_df = evaluate_models(emb_data, labels, emb_name)
    all_results.append(res_df)

final_results_df = pd.concat(all_results, ignore_index=True)
print("Evaluation Complete.")
final_results_df.sort_values(by='F1-score', ascending=False).head(10)

Starting Model Evaluation...
Evaluating models with TF-IDF...


Evaluating models with Word2Vec...
Evaluating models with GloVe...
Evaluating models with GloVe...
Evaluating models with Sentence-BERT...
Evaluating models with Sentence-BERT...
Evaluating models with CodeBERT...
Evaluating models with CodeBERT...
Evaluating models with RoBERTa...
Evaluating models with RoBERTa...
Evaluating models with Jina-V2...
Evaluating models with Jina-V2...
Evaluating models with Nomic...
Evaluating models with Nomic...
Evaluation Complete.
Evaluation Complete.


,Model,Embedding,Accuracy,Precision,Recall,F1-score,Time (s)
2,SVM,TF-IDF,0.723776,0.726419,0.723776,0.716944,4.457238
0,Logistic Regression,TF-IDF,0.720280,0.728077,0.720280,0.712002,35.617671
49,Logistic Regression,Nomic,0.688811,0.715612,0.688811,0.695305,152.730168
5,XGBoost,TF-IDF,0.699301,0.723104,0.699301,0.694366,13.917635
6,KNN,TF-IDF,0.692308,0.726394,0.692308,0.682468,0.037995
3,Random Forest,TF-IDF,0.695804,0.691751,0.695804,0.670939,0.644978
4,Gradient Boosting,TF-IDF,0.657343,0.695392,0.657343,0.660203,88.771072
51,SVM,Nomic,0.671329,0.692755,0.671329,0.654642,1.299852
1,Naive Bayes,TF-IDF,0.639860,0.676240,0.639860,0.646436,0.111010
35,Logistic Regression,RoBERTa,0.650350,0.666979,0.650350,0.644080,764.053809


In [12]:
# Deep Learning Setup
from collections import Counter
from sklearn.preprocessing import LabelEncoder

# Tokenization and Padding for DL
MAX_LEN = 200
VOCAB_SIZE = 5000

# Build Vocab
all_words = [word for code in df['cleaned_code'] for word in code.split()]
word_counts = Counter(all_words)
common_words = word_counts.most_common(VOCAB_SIZE - 2) # -2 for PAD and UNK
vocab = {word: i+2 for i, (word, _) in enumerate(common_words)}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

def encode_text(text, vocab, max_len):
    tokens = text.split()
    encoded = [vocab.get(word, 1) for word in tokens]
    if len(encoded) < max_len:
        encoded += [0] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]
    return encoded

X_seq = np.array([encode_text(text, vocab, MAX_LEN) for text in df['cleaned_code']])
le = LabelEncoder()
y_enc = le.fit_transform(df['label'])
NUM_CLASSES = len(le.classes_)

# PyTorch Dataset
class CodeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Split
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(X_seq, y_enc, test_size=0.2, random_state=42)
train_ds = CodeDataset(X_train_seq, y_train_seq)
test_ds = CodeDataset(X_test_seq, y_test_seq)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16)

# Training Function
def train_model(model, train_loader, test_loader, epochs=5, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
    # Evaluate
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y_batch.numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    return acc, f1

print("DL Data Prepared.")

DL Data Prepared.


In [13]:
# 1. Simple Feed-Forward (using Embeddings layer)
class SimpleNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super(SimpleNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x, dim=1) # Average pooling
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# 2. CNN Text Classifier
class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super(CNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv1 = nn.Conv1d(embed_dim, 128, kernel_size=5)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        x = self.embedding(x) # (batch, seq, embed)
        x = x.permute(0, 2, 1) # (batch, embed, seq)
        x = self.conv1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.pool(x).squeeze(2)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# 3. LSTM / BiLSTM
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, bidirectional=False):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=bidirectional, dropout=0.3 if bidirectional else 0)
        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), num_classes)
        
    def forward(self, x):
        x = self.embedding(x)
        _, (hn, _) = self.lstm(x)
        if self.lstm.bidirectional:
            out = torch.cat((hn[-2], hn[-1]), dim=1)
        else:
            out = hn[-1]
        x = self.fc(out)
        return x

# Update training function to include scheduler
def train_model_with_scheduler(model, train_loader, test_loader, epochs=10, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        scheduler.step()
            
    # Evaluate
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y_batch.numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    return acc, f1

# Train DL Models
dl_results = []
models_dl = {
    'Simple NN': SimpleNN(VOCAB_SIZE, 100, 64, NUM_CLASSES),
    'CNN': CNNClassifier(VOCAB_SIZE, 100, NUM_CLASSES),
    'LSTM': LSTMClassifier(VOCAB_SIZE, 100, 64, NUM_CLASSES, bidirectional=False),
    'BiLSTM': LSTMClassifier(VOCAB_SIZE, 100, 64, NUM_CLASSES, bidirectional=True)
}

print("Training DL Models...")
for name, model in models_dl.items():
    print(f"Training {name}...")
    start_time = time.time()
    acc, f1 = train_model_with_scheduler(model, train_loader, test_loader, epochs=10)
    end_time = time.time()
    
    dl_results.append({
        'Model': name,
        'Embedding': 'Learned (End-to-End)',
        'Accuracy': acc,
        'Precision': '-', # Simplified for DL loop
        'Recall': '-',
        'F1-score': f1,
        'Time (s)': end_time - start_time
    })

dl_results_df = pd.DataFrame(dl_results)
dl_results_df

Training DL Models...
Training Simple NN...
Training CNN...
Training CNN...
Training LSTM...
Training LSTM...
Training BiLSTM...


,Model,Embedding,Accuracy,Precision,Recall,F1-score,Time (s)
0,Simple NN,Learned (End-to-End),0.489510,-,-,0.457034,7.782984
1,CNN,Learned (End-to-End),0.465035,-,-,0.441950,17.589372
2,LSTM,Learned (End-to-End),0.213287,-,-,0.179433,37.996925
3,BiLSTM,Learned (End-to-End),0.402098,-,-,0.380378,68.807901


In [14]:
# Explainability
print("Running Explainability Analysis...")

# 1. ML Explainability (LIME)
# Train a specific model for explanation (Logistic Regression + TF-IDF)
explainer_model = LogisticRegression(max_iter=1000)
explainer_model.fit(X_tfidf, labels)

# LIME
from lime.lime_text import LimeTextExplainer
from sklearn.pipeline import make_pipeline

# Create a pipeline for LIME
pipe = make_pipeline(tfidf_vectorizer, explainer_model)

class_names = list(explainer_model.classes_)
explainer = LimeTextExplainer(class_names=class_names)

# Explain a sample
idx = 0
text_instance = df['cleaned_code'].iloc[idx]
print(f"Explaining instance {idx} (True Label: {df['label'].iloc[idx]})")

exp = explainer.explain_instance(text_instance, pipe.predict_proba, num_features=10)
print("LIME Explanation for ML (Logistic Regression):")
print(exp.as_list())

# 2. DL Explainability (LIME)
# We need a wrapper for the PyTorch model
model_to_explain = models_dl['Simple NN']
model_to_explain.eval()

def predict_proba_dl(texts):
    # Encode
    encoded_texts = [encode_text(t, vocab, MAX_LEN) for t in texts]
    tensor_texts = torch.tensor(encoded_texts, dtype=torch.long)
    
    with torch.no_grad():
        outputs = model_to_explain(tensor_texts)
        probs = torch.softmax(outputs, dim=1).numpy()
    return probs

print("\nExplaining DL Model (Simple NN)...")
explainer_dl = LimeTextExplainer(class_names=le.classes_)
exp_dl = explainer_dl.explain_instance(text_instance, predict_proba_dl, num_features=10)
print("LIME Explanation for DL (Simple NN):")
print(exp_dl.as_list())

# 3. SHAP (ML)
import shap
# SHAP for linear models
# We use a small background dataset for speed
print("\nRunning SHAP for ML...")
masker = shap.maskers.Text(r"\W+") 
# Use a linear explainer on the model directly (faster than KernelExplainer)
# But we need the coefficients. 
# Let's use the generic Explainer with the pipeline, but on a very small sample.
explainer_shap = shap.Explainer(pipe.predict_proba, masker)
shap_values = explainer_shap(df['cleaned_code'].iloc[:2]) # Explain first 2 samples

print("SHAP values generated for 2 samples.")
# We can't easily print the plot, but we can print the values
print("SHAP Values shape:", shap_values.shape)


Running Explainability Analysis...
Explaining instance 0 (True Label: Advanced LLM Prompting)
LIME Explanation for ML (Logistic Regression):
[(np.str_('prompt'), 0.005539260577444622), (np.str_('topic'), -0.002972498485433683), (np.str_('self'), -0.0012862997233064474), (np.str_('json'), -0.0010304959007239939), (np.str_('template'), -0.0008480467142461625), (np.str_('the'), -0.0007995379807300769), (np.str_('str'), -0.0007718512219520106), (np.str_('if'), -0.000726500580205726), (np.str_('find'), 0.0003570288853445186), (np.str_('concept'), 0.00033750520937445976)]

Explaining DL Model (Simple NN)...
LIME Explanation for DL (Simple NN):
[(np.str_('concept'), 0.015591409072343384), (np.str_('prompt'), 0.01465238531500534), (np.str_('the'), -0.014191472217946121), (np.str_('question'), -0.009736182997874245), (np.str_('find'), -0.009685493103551853), (np.str_('is'), 0.009337939179134554), (np.str_('about'), 0.00884386164053717), (np.str_('answer'), -0.008821933636045562), (np.str_('topi

In [16]:
# Comparison Table
final_df = pd.concat([final_results_df, dl_results_df], ignore_index=True)
final_df = final_df.sort_values(by='F1-score', ascending=False)
print("Final Comparison Table:")
print(final_df)

# Generate Report
report_content = f"""
# AI Design Pattern Classification Report

## 1. Introduction
This report summarizes the classification of AI Design Patterns using various embedding techniques and machine learning models.

## 2. Dataset
- **Source**: {DATASET_PATH}
- **Samples**: {len(df)}
- **Classes**: {len(df['label'].unique())} ({', '.join(df['label'].unique())})

## 3. Embeddings Evaluated
- Traditional: TF-IDF, Word2Vec, GloVe
- Modern: Sentence-BERT, CodeBERT, RoBERTa, Jina-V2 (where available)

## 4. Models Evaluated
- **ML**: Logistic Regression, Naive Bayes, SVM, Random Forest, Gradient Boosting, XGBoost, KNN
- **DL**: Simple NN, CNN, LSTM, BiLSTM

## 5. Results Summary
Top 5 Performing Models:
{final_df[['Model', 'Embedding', 'Accuracy', 'F1-score']].head(5).to_markdown(index=False)}

## 6. Explainability
LIME analysis was performed to identify key tokens contributing to classification.

## 7. Conclusion
The best performing model was **{final_df.iloc[0]['Model']}** with **{final_df.iloc[0]['Embedding']}** embedding, achieving an F1-score of **{final_df.iloc[0]['F1-score']:.4f}**.
"""

report_path = "/root/AI-Pattern-Mining-Project/experiments-llm/AI_Pattern_Classification_Report.md"
with open(report_path, "w") as f:
    f.write(report_content)

print(f"Report generated at {report_path}")

Final Comparison Table:
                  Model             Embedding  Accuracy Precision    Recall  \
2                   SVM                TF-IDF  0.723776  0.726419  0.723776   
0   Logistic Regression                TF-IDF  0.720280  0.728077   0.72028   
49  Logistic Regression                 Nomic  0.688811  0.715612  0.688811   
5               XGBoost                TF-IDF  0.699301  0.723104  0.699301   
6                   KNN                TF-IDF  0.692308  0.726394  0.692308   
3         Random Forest                TF-IDF  0.695804  0.691751  0.695804   
4     Gradient Boosting                TF-IDF  0.657343  0.695392  0.657343   
51                  SVM                 Nomic  0.671329  0.692755  0.671329   
1           Naive Bayes                TF-IDF  0.639860   0.67624   0.63986   
35  Logistic Regression               RoBERTa  0.650350  0.666979   0.65035   
42  Logistic Regression               Jina-V2  0.636364  0.664826  0.636364   
44                  SVM     